In [1]:
import pystac_client
import planetary_computer
import odc.stac

In [2]:
# map tool where I can draw a box and it puts the coordinates in a python list

In [ ]:
bbox = (-79.63013616915275, 39.88, -79.62286245374449, 39.90174491151549)

In [ ]:
# jupyter notebook cell to draw bbox on a map and capture it in a variable
from ipyleaflet import DrawControl, Map, basemaps

# 1. Initialize the interactive map
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(40.015, -105.270),  # Latitude, Longitude center
    zoom=10,
)

# 2. Define a global variable to store your bounding box coordinates
bbox_coords = None


# 3. Define a callback function to handle draw events
def handle_draw(target, action, geo_json):
    global bbox_coords
    # Check if the drawn shape is a Polygon/Rectangle
    if geo_json["geometry"]["type"] == "Polygon" and action == "created":
        # Extract the coordinate ring (outer boundary)
        coordinates = geo_json["geometry"]["coordinates"][0]

        # Calculate bounding box bounds: [min_lon, min_lat, max_lon, max_lat]
        lons = [pt[0] for pt in coordinates]
        lats = [pt[1] for pt in coordinates]
        bbox_coords = [min(lons), min(lats), max(lons), max(lats)]

        print(f"Captured BBOX [min_lon, min_lat, max_lon, max_lat]:")
        print(bbox_coords)


# 4. Configure the DrawControl (keep only the rectangle tool active)
draw_control = DrawControl()
draw_control.polyline = {}
draw_control.polygon = {}
draw_control.circlemarker = {}
draw_control.rectangle = {
    "shapeOptions": {"fillColor": "#3388ff", "color": "#3388ff", "fillOpacity": 0.2}
}

# Attach the callback function to the draw control
draw_control.on_draw(handle_draw)

# Add the drawing tools to the map
m.add(draw_control)

# Display the map widget
m

Map(center=[40.015, -105.27], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [ ]:
from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info

utm_crs_list = query_utm_crs_info(
    datum_name="WGS 84",
    area_of_interest=AreaOfInterest(
        west_lon_degree=bbox_coords[0],
        south_lat_degree=bbox_coords[1],
        east_lon_degree=bbox_coords[2],
        north_lat_degree=bbox_coords[3],
    ),
)
utm_crs = f"{utm_crs_list[0].auth_name}:{utm_crs_list[0].code}"  # EPSG code as string

In [44]:
bbox_coords

[-105.31168, 39.999545, -105.305758, 40.003292]

In [35]:
# 1. Search MPC STAC catalog for NAIP over your AOI
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# get utm zone relevant for the bbox

# bbox= (144.6909970943081,13.40925238659056,144.6939145736856,13.41447897567025)
# (-95.16158000899202, 44.62473082801966, -95.15528522624939, 44.62934880921187)
# [-105.313568,40.016835,-105.304470,40.022488]  # west, south, east, north

search = catalog.search(
    collections=["naip"],
    bbox=bbox_coords,
    datetime="2020/2026",
)
items = list(search.items())

import rioxarray
from rasterio.windows import from_bounds

href = planetary_computer.sign(items[0].assets["image"].href)
da = rioxarray.open_rasterio(href, masked=True)

# clip to bbox in the tile's native CRS
# your bbox is in lon/lat (EPSG:4326), so reproject bounds to match da's CRS first
import rasterio.warp

bbox_native = rasterio.warp.transform_bounds("EPSG:4326", da.rio.crs, *bbox_coords)

# da_clipped = da.rio.clip_box(*bbox_native)

# reproject to UTM zone 17N (EPSG:26917) at 1m resolution
da_final = da.rio.clip_box(*bbox_native).rio.reproject(utm_crs, resolution=1.0)

In [36]:
box_width = bbox_native[2] - bbox_native[0]
box_height = bbox_native[3] - bbox_native[1]
box_width, box_height

(506.93140952673275, 417.63232234027237)

In [37]:
import matplotlib.pyplot as plt

# 1. Transpose from (band, y, x) to (y, x, band) for image saving
rgb = da_final.sel(band=[1, 2, 3])  # rioxarray bands are 1-indexed
rgb_norm = rgb / rgb.max()

img_array = rgb_norm.transpose("y", "x", "band").values

# 2. Save using Matplotlib
plt.imsave("output_image.png", img_array)

In [ ]:
# 2. Load directly into an xarray Dataset, reprojected/resampled to your target CRS + resolution
ds = odc.stac.load(
    items,
    bands=["image"],
    crs=utm_crs,  # match your 3DEP/DEM CRS (e.g. UTM zone for PA)
    resolution=1.0,  # snap to whatever grid you're using for 3DEP products
    bbox=bbox_coords,
)

In [41]:
ds

<xarray.Dataset> Size: 3MB
Dimensions:      (y: 419, x: 508, time: 3)
Coordinates:
  * y            (y) float64 3kB 4.428e+06 4.428e+06 ... 4.428e+06 4.428e+06
  * x            (x) float64 4kB 4.734e+05 4.734e+05 ... 4.739e+05 4.739e+05
  * time         (time) datetime64[us] 24B 2021-07-26T16:00:00 ... 2023-09-25...
    spatial_ref  int32 4B 32613
Data variables:
    image        (time, y, x) float32 3MB 75.0 80.0 90.0 88.0 ... 44.0 44.0 43.0